# Offline Reinforcement Learning and Decision Transformers

Every algorithm in this course has assumed that the agent can **interact** with the environment: try an action, see what happens, and learn from it. The off-policy methods of Chapters 07 and 10 reuse past experience, but they still collect that experience themselves, and they keep collecting it while they learn. In many of the applications where reinforcement learning would be most valuable, this assumption does not hold. A hospital has records of how patients were treated and how they fared, but will not let an algorithm experiment on them. A fleet of cars has millions of kilometres of driving logs, but a learning controller cannot be allowed to explore. A recommender system, a dialogue system, an industrial process: in all these cases there is a large **fixed dataset** of past behaviour, produced by humans or by previous controllers, and no possibility of trying things out.

**Offline reinforcement learning** (also called *batch* reinforcement learning) is the problem of learning the best possible policy from such a dataset, without any further interaction. It is, in a sense, the natural meeting point of reinforcement learning and the data-driven approach of supervised learning: a static dataset, a model trained on it, and the hope that the model generalises. But the analogy is treacherous, and the first lesson of this chapter is that the off-policy methods we already know, which seem designed for exactly this situation, **fail** when they are cut off from the environment, for a reason that is specific to reinforcement learning. The second lesson is how to fix this, by making the value estimates **conservative** where the data say nothing. The third is an altogether different approach, the **Decision Transformer**, which abandons value functions and treats reinforcement learning as a **sequence modelling** problem, with the same architecture that powers large language models.

The chapter proceeds as follows. We define the offline setting and build three datasets of Lunar Lander trajectories of different quality. We start from the simplest idea, imitating the data by supervised learning, and see what it can and cannot do. We then show, on the same datasets, why an off-policy value-based method diverges, and derive Conservative Q-Learning as a remedy. Finally, after a compact introduction to the transformer architecture, we implement a Decision Transformer and see how conditioning on a desired return turns a dataset of mixed quality into a controllable policy.

## The Offline Setting

In the offline setting the agent receives a dataset $\mathcal{D}$ of transitions, or of whole trajectories,

$\displaystyle \mathcal{D} = \big\{ (s_i, a_i, r_i, s'_i, d_i) \big\}_{i=1}^{N}$

with $d_i$ the terminal flag of Chapter 07, collected by one or more **behaviour policies** $\pi_\beta$ that are unknown and may be of any quality: an expert, a mediocre controller, a random exploration, or a mixture. The objective is still the expected return, but the only information available about the environment is what the dataset contains. The table summarises how this differs from the settings we have seen:

| Setting | Who collects the data | Does learning change the data? | Chapters |
|---|---|---|---|
| On-policy | the current policy | yes, every update | 08, 09, 11 |
| Off-policy (with replay) | past versions of the policy, plus exploration | yes, the memory keeps filling | 07, 10 |
| Offline | an unknown behaviour policy, once and for all | **no** | 12 |

The last column is the crux. In the off-policy setting, if the agent's value estimates are wrong somewhere, the agent will sooner or later go there, observe the truth and correct them. In the offline setting, **the dataset is the only feedback the agent will ever receive**: a mistake about a state-action pair that is not in the data can never be corrected, because the pair will never be observed. This single fact is responsible for most of the difficulty of offline reinforcement learning.

A second difficulty is **evaluation**. In this chapter we evaluate the policies we learn by running them in the simulator, as we have always done; in a real offline problem this is precisely what is not allowed, and estimating the value of a new policy from logged data, a problem known as *off-policy evaluation*, is a research area of its own. We also evaluate against the natural reference points: the behaviour policy that generated each dataset, and the best policy we know for the environment.

## Building the Datasets

Offline reinforcement learning is usually studied on standard benchmark datasets, collected by trained agents of various qualities on simulated tasks. We can build datasets of the same kind ourselves. Gymnasium ships a hand-written **heuristic controller** for Lunar Lander, a proportional-derivative controller on the angle and the altitude, which lands reliably: it will play the role of the expert. By corrupting its actions with random ones we obtain policies of lower quality: with probability $\epsilon$ the action is random, otherwise it is the heuristic's. We generate three datasets, each from a few hundred episodes:

- **expert**: the heuristic controller ($\epsilon = 0$);
- **medium**: the heuristic with $\epsilon = 0.2$, which lands sometimes and crashes sometimes;
- **random**: uniformly random actions ($\epsilon = 1$).

The datasets are stored as flat arrays of transitions, with the boundaries of the episodes recorded separately, since some of the methods below need whole trajectories:

In [ ]:
import gymnasium
import numpy as np
import torch
import random
import matplotlib.pyplot as plt

from gymnasium.envs.box2d.lunar_lander import heuristic

lunar_lander = gymnasium.make('LunarLander-v3', render_mode="rgb_array")

state_size = lunar_lander.observation_space.shape[0]
action_size = lunar_lander.action_space.n

print('State size: ', state_size)
print('Action size: ', action_size)

In [ ]:
def expert_pi(state):
    # the hand-written controller shipped with the environment
    return int(heuristic(lunar_lander.unwrapped, state))


def noisy_expert_pi(state, epsilon=0.2):
    # the expert's action, replaced by a random one with probability epsilon
    if random.random() < epsilon:
        return lunar_lander.action_space.sample()
    return expert_pi(state)


def random_pi(state):
    return lunar_lander.action_space.sample()

In [ ]:
def collect_dataset(env, policy, n_episodes, seed):

    # set the seeds for reproducibility
    random.seed(seed)
    np.random.seed(seed)
    env.reset(seed=seed)
    env.action_space.seed(seed)

    states, actions, rewards, next_states, terminals = [], [], [], [], []
    episode_starts, episode_returns = [], []

    for episode in range(n_episodes):

        state, _ = env.reset()
        done = False
        episode_starts.append(len(states))
        episode_return = 0.0

        while not done:
            action = policy(state)
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            states.append(state)
            actions.append(action)
            rewards.append(reward)
            next_states.append(next_state)
            terminals.append(terminated)

            state = next_state
            episode_return += reward

        episode_returns.append(episode_return)

    dataset = {
        "states": np.array(states, dtype=np.float32),
        "actions": np.array(actions, dtype=np.int64),
        "rewards": np.array(rewards, dtype=np.float32),
        "next_states": np.array(next_states, dtype=np.float32),
        "terminals": np.array(terminals, dtype=np.float32),
        "episode_starts": np.array(episode_starts + [len(states)]),
        "episode_returns": np.array(episode_returns, dtype=np.float32),
    }

    return dataset

In [ ]:
expert_dataset = collect_dataset(lunar_lander, expert_pi, n_episodes=300, seed=12)
medium_dataset = collect_dataset(lunar_lander, noisy_expert_pi, n_episodes=300, seed=34)
random_dataset = collect_dataset(lunar_lander, random_pi, n_episodes=300, seed=56)

A dataset is characterised by its size and, above all, by the **distribution of the returns** of its episodes, which tells us how good the behaviour policy was and how diverse its outcomes:

In [ ]:
def describe_dataset(name, dataset):
    returns = dataset["episode_returns"]
    print(f"{name:8s}: {len(dataset['states']):7d} transitions, {len(returns):4d} episodes, "
          f"return mean {returns.mean():7.1f}, std {returns.std():6.1f}, min {returns.min():7.1f}, max {returns.max():7.1f}")

describe_dataset("expert", expert_dataset)
describe_dataset("medium", medium_dataset)
describe_dataset("random", random_dataset)

In [ ]:
plt.figure(figsize=(12, 5))
plt.title("Distribution of the episode returns in the three datasets")
plt.xlabel("Episode return")
plt.ylabel("Episodes")

for name, dataset, color in (("expert", expert_dataset, "green"), ("medium", medium_dataset, "orange"), ("random", random_dataset, "gray")):
    plt.hist(dataset["episode_returns"], bins=40, alpha=0.5, color=color, label=name)

plt.legend()
plt.show()

The expert episodes are concentrated at high returns; the random ones at strongly negative returns; the medium dataset is the interesting one, spread over the whole range from crashes to good landings. Note what the datasets do **not** contain. The expert dataset never shows what happens after a bad action, because the expert never takes one; the random dataset never shows a landing. Each dataset covers a region of the state-action space, and nothing outside it. The evaluation function and the visualisation function are those of the previous chapters:

In [ ]:
def evaluate(env, policy, episodes=1):

    # collect total rewards per episode
    rewards = []

    for episode in range(episodes):

        # reset the environment
        state, _ = env.reset()
        done = False
        total_reward = 0.0

        # run one episode
        while not done:
            action = policy(state)
            state, reward, terminated, truncated, _ = env.step(action)
            total_reward += reward
            done = terminated or truncated

        rewards.append(total_reward)

    # return the average reward over episodes
    return float(np.mean(rewards))

In [ ]:
from IPython.display import clear_output

def show_policy(env, policy, n_episodes=1, max_steps=1000):

    for episode in range(n_episodes):

        # reset the environment
        state, _ = env.reset()
        done = False
        step = 0

        # run the episode until done or max_steps
        while not done and step < max_steps:

            # make a step
            action = policy(state)
            state, _, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            # render the environment
            clear_output(wait=True)
            plt.imshow(env.render())
            plt.axis("off")
            plt.title(f"Episode {episode+1}, Step {step}")
            plt.show()
            plt.close()

            step += 1

We record the scores of the three behaviour policies as reference points:

In [ ]:
lunar_lander.reset(seed=34)
lunar_lander.action_space.seed(34)

expert_score = evaluate(lunar_lander, expert_pi, episodes=20)
medium_score = evaluate(lunar_lander, noisy_expert_pi, episodes=20)
random_score = evaluate(lunar_lander, random_pi, episodes=20)

print("Expert score", expert_score)
print("Medium score", medium_score)
print("Random score", random_score)

## Behaviour Cloning

The simplest thing one can do with a dataset of state-action pairs is to **imitate** it: treat the states as inputs, the actions as labels, and train a classifier. This is **behaviour cloning**, and it is ordinary supervised learning. The policy network of Chapter 08, a network with one logit per action, is trained by maximum likelihood on the dataset,

$\displaystyle \mathcal{L}(\theta) = - \frac{1}{N} \sum_{i=1}^{N} \log \pi_\theta(a_i \mid s_i)$

which is the cross-entropy loss of a classifier. No rewards, no values, no bootstrapping: behaviour cloning does not even look at the reward column of the dataset. At test time the learned policy is run greedily, taking the most probable action.

Two things make behaviour cloning weaker than it looks. First, by construction it can at best **reproduce the behaviour policy**: cloning a mediocre dataset gives a mediocre policy, and the rewards, which would tell good actions from bad ones, are ignored. Second, its errors **compound**. A supervised classifier is accurate on the distribution it was trained on; but as soon as the cloned policy makes a mistake, it drifts to states the expert never visited, where it has no idea what to do, makes further mistakes, and drifts further. In a horizon of $T$ steps, a per-step error probability $\eta$ can produce a total cost growing like $\eta T^2$ rather than $\eta T$, a result due to Ross and Bagnell (2010). The remedy, when the expert is available, is to query it in the states the learner visits (the DAgger algorithm of Ross, Gordon and Bagnell, 2011); offline, no such remedy exists.

In [ ]:
def create_policy_network(state_size, action_size, first_hidden_layer=256, second_hidden_layer=256):

    dnn = torch.nn.Sequential(
        torch.nn.Linear(state_size, first_hidden_layer),
        torch.nn.ReLU(),
        torch.nn.Linear(first_hidden_layer, second_hidden_layer),
        torch.nn.ReLU(),
        # output layer: one preference (logit) per action
        torch.nn.Linear(second_hidden_layer, action_size),
    )

    return dnn


def sample_batch(dataset, batch_size=256):

    # a mini-batch of transitions sampled uniformly from the dataset
    indices = np.random.randint(0, len(dataset["states"]), size=batch_size)

    return (
        torch.as_tensor(dataset["states"][indices]),
        torch.as_tensor(dataset["actions"][indices]),
        torch.as_tensor(dataset["rewards"][indices]),
        torch.as_tensor(dataset["next_states"][indices]),
        torch.as_tensor(dataset["terminals"][indices]),
    )

In [ ]:
def behaviour_cloning(dataset, steps=20_000, batch_size=256, learning_rate=0.0003):

    # a policy network and its optimizer
    pi = create_policy_network(state_size, action_size)
    optimizer = torch.optim.Adam(pi.parameters(), lr=learning_rate)

    for step in range(1, steps + 1):

        states, actions, _, _, _ = sample_batch(dataset, batch_size)

        # cross-entropy: minus the log-probability of the actions in the dataset
        loss = torch.nn.functional.cross_entropy(pi(states), actions)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if step % 1000 == 0:
            print(f"Step {step:06}, loss {loss.item():.4f}", end="\r", flush=True)

    # the greedy policy of the trained network
    def bc_pi(state):
        with torch.no_grad():
            return int(torch.argmax(pi(torch.as_tensor(state, dtype=torch.float32))).item())

    return bc_pi

We clone each of the three datasets and evaluate the result:

In [ ]:
torch.manual_seed(34)
np.random.seed(34)
random.seed(34)

bc_scores = {}
for name, dataset in (("expert", expert_dataset), ("medium", medium_dataset), ("random", random_dataset)):
    bc_pi = behaviour_cloning(dataset)
    bc_scores[name] = evaluate(lunar_lander, bc_pi, episodes=20)
    print(f"\nBehaviour cloning on the {name} dataset: score {bc_scores[name]:.1f}")

On the expert dataset behaviour cloning works well: the controller is a simple deterministic function of the state, the network learns it, and the cloned policy lands almost as reliably as the expert. On the random dataset it learns nothing useful, as it should. The medium dataset deserves a closer look. The behaviour policy was the expert's action $80\%$ of the time and a random action $20\%$ of the time; the classifier learns, in every state, a distribution with most of its mass on the expert's action, and taking the most probable action **filters the noise out**: the cloned policy can be considerably better than the behaviour policy that generated the data. This is a happy accident of how this dataset was made, where the "bad" actions are unbiased noise around a good policy. Had the medium data been produced by a genuinely different, consistently mediocre controller, cloning would have reproduced its mediocrity faithfully: behaviour cloning has no way of knowing which actions in the data were good, because it never looks at the rewards. Using the rewards is what reinforcement learning is about, and it is what we try next.

## Why Off-Policy Methods Fail Offline

Off-policy value-based methods seem made for the offline setting. DQN learns from a replay memory of transitions, sampled uniformly, with no requirement that they come from the current policy; a dataset is just a replay memory that is never refilled. Let us run exactly that: the Double DQN of Chapter 07, with a target network, trained on mini-batches sampled from a fixed dataset, with the policy derived greedily from the learned action values. To monitor what happens inside, we track two quantities on a fixed sample of dataset states: the value the network assigns to the **actions actually taken** in the data, $Q(s,a)$, and the value it assigns to the **greedy action**, $\max_a Q(s,a)$, which is what the learned policy believes it will obtain. We compare both with the average **discounted return actually observed** after those state-action pairs in the data, a Monte Carlo estimate of $q_{\pi_\beta}(s,a)$, which is what the value of the behaviour policy looks like. The greedy value should, if anything, be *a little* above this reference, since the greedy policy improves on the behaviour policy; it should certainly not be far above the maximum return that the environment can produce.

In [ ]:
def discounted_returns(dataset, gamma=0.99):

    # discounted return-to-go of every transition, computed backwards within each episode
    returns = np.zeros_like(dataset["rewards"])
    starts = dataset["episode_starts"]

    for episode in range(len(starts) - 1):
        future_return = 0.0
        for t in reversed(range(starts[episode], starts[episode + 1])):
            future_return = dataset["rewards"][t] + gamma * future_return
            returns[t] = future_return

    return returns

In [ ]:
def offline_dqn(dataset, steps=60_000, batch_size=256, learning_rate=0.0003, gamma=0.99, target_update_steps=1000, conservative_alpha=0.0):

    # online and target Q-networks and the optimizer
    q = create_policy_network(state_size, action_size)
    target_q = create_policy_network(state_size, action_size)
    target_q.load_state_dict(q.state_dict())
    optimizer = torch.optim.Adam(q.parameters(), lr=learning_rate)

    # a fixed sample of dataset pairs to monitor the Q-values, and their observed discounted returns
    monitor = np.random.randint(0, len(dataset["states"]), size=2000)
    monitor_states = torch.as_tensor(dataset["states"][monitor])
    monitor_actions = torch.as_tensor(dataset["actions"][monitor])
    monitor_returns = discounted_returns(dataset, gamma)[monitor].mean()

    q_values_log = []    # mean Q of the dataset actions
    max_q_log = []       # mean Q of the greedy actions

    for step in range(1, steps + 1):

        states, actions, rewards, next_states, terminals = sample_batch(dataset, batch_size)

        # Double DQN target: the online network selects, the target network evaluates
        with torch.no_grad():
            best_actions = q(next_states).argmax(dim=1)
            next_q = target_q(next_states).gather(1, best_actions.unsqueeze(1)).squeeze(1)
            target = rewards + gamma * (1.0 - terminals) * next_q

        # Q-values of the dataset actions
        q_all = q(states)
        q_current = q_all.gather(1, actions.unsqueeze(1)).squeeze(1)

        # mean squared TD error
        loss = (target - q_current).pow(2).mean()

        # conservative penalty (CQL): push down the soft maximum over all actions, push up the dataset actions
        if conservative_alpha > 0:
            loss = loss + conservative_alpha * (torch.logsumexp(q_all, dim=1) - q_current).mean()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # hard update of the target network
        if step % target_update_steps == 0:
            target_q.load_state_dict(q.state_dict())

        # monitor the Q-values on the fixed sample
        if step % 1000 == 0:
            with torch.no_grad():
                monitor_q = q(monitor_states)
                mean_q = monitor_q.gather(1, monitor_actions.unsqueeze(1)).mean().item()
                max_q = monitor_q.max(dim=1).values.mean().item()
            q_values_log.append(mean_q)
            max_q_log.append(max_q)
            print(f"Step {step:06}, loss {loss.item():8.2f}, Q of dataset actions {mean_q:8.2f}, Q of greedy actions {max_q:8.2f} (dataset return {monitor_returns:7.2f})", end="\r", flush=True)

    # the greedy policy of the trained network
    def dqn_pi(state):
        with torch.no_grad():
            return int(torch.argmax(q(torch.as_tensor(state, dtype=torch.float32))).item())

    return dqn_pi, np.array(q_values_log), np.array(max_q_log), monitor_returns

We train the offline agent on the three datasets and plot the evolution of its Q-values against the returns observed in the data:

In [ ]:
torch.manual_seed(34)
np.random.seed(34)
random.seed(34)

dqn_scores = {}
dqn_logs = {}
for name, dataset in (("expert", expert_dataset), ("medium", medium_dataset), ("random", random_dataset)):
    dqn_pi, q_log, max_q_log, data_return = offline_dqn(dataset)
    dqn_scores[name] = evaluate(lunar_lander, dqn_pi, episodes=20)
    dqn_logs[name] = (q_log, max_q_log, data_return)
    print(f"\nOffline DQN on the {name} dataset: score {dqn_scores[name]:.1f}, Q of dataset actions {q_log[-1]:.1f}, Q of greedy actions {max_q_log[-1]:.1f}, dataset return {data_return:.1f}")

In [ ]:
def plot_q_values(logs, title):

    plt.figure(figsize=(12, 5))
    plt.title(title)
    plt.xlabel("Gradient steps")
    plt.ylabel("Mean value")

    for name, color in (("expert", "green"), ("medium", "orange"), ("random", "gray")):
        q_log, max_q_log, data_return = logs[name]
        steps = (np.arange(len(q_log)) + 1) * 1000
        plt.plot(steps, max_q_log, color=color, linewidth=2, label=f"{name}: Q of the greedy actions")
        plt.plot(steps, q_log, color=color, linewidth=1, linestyle=":", label=f"{name}: Q of the dataset actions")
        plt.axhline(data_return, color=color, linestyle="--", linewidth=1, label=f"{name}: observed return")

    plt.legend(fontsize=8, ncol=3)
    plt.show()


plot_q_values(dqn_logs, "Offline DQN: estimated values vs. observed discounted returns")

### Extrapolation Error

The estimated values **run away** from the observed returns: after a few tens of thousands of gradient steps they exceed them several times over, and they are still accelerating when training stops. The value of the greedy actions, the one the learned policy believes in, leads the way, and the values of the dataset's own actions, which are the pairs the network is trained on, follow closely, pulled up by the inflated targets. The greedy policies derived from the networks are poor, worse than behaviour cloning on the same data and, on the expert dataset, worse than the random policy. Nothing is wrong with the implementation: this is the normal behaviour of bootstrapped value learning when it is cut off from the environment, and it has a name, **extrapolation error** (Fujimoto, Meger and Precup, 2019). The mechanism is the following. The target of every update contains a maximum over actions in the next state,

$\displaystyle y = r + \gamma \max_{a'} Q(s', a')$

(in Double DQN the maximum is taken by the online network and evaluated by the target network, which changes nothing in the argument), and the maximum ranges over **all** actions, including those that the behaviour policy never took in states like $s'$. For those actions the network's estimate is pure extrapolation: it was never trained on them, so its value is whatever the function approximator happens to produce there. Among several such arbitrary values, the maximum is likely to be an **overestimate**; this is the maximisation bias of Chapter 05 again, but with a crucial difference. Online, an overestimated action is tempting, the agent tries it, observes a disappointing reward, and the estimate is corrected: exploration is the antidote. Offline, the overestimated action is never tried, its value is never corrected, and worse, it is **propagated**: it enters the target of the previous state, inflates it, and the inflated value enters the target before that. Errors on out-of-distribution actions compound backwards through the bootstrapping, update after update, with nothing to stop them. Double DQN and the target network, which contain the problem online, only slow it down.

The failure is worst where the dataset covers the state-action space least, which is why the random and medium datasets suffer more than the expert one, and it gets worse with longer training, since each target update propagates the errors one step further. This is the central difficulty of offline reinforcement learning: **a value-based method must query its estimates at actions the data do not support, and has no way to learn that those estimates are wrong.**

## Conservative Q-Learning (CQL)

If the problem is that the values of unseen actions are too high, the remedy is to make them **low** on purpose. [**Conservative Q-Learning**](./papers/2020%20-%20Conservative%20Q-Learning%20for%20Offline%20Reinforcement%20Learning.pdf), introduced by Kumar and colleagues in 2020, adds to the usual TD loss a regulariser that pushes the Q-values **down** for actions the learned policy might choose, and **up** for the actions that appear in the dataset:

$\displaystyle \mathcal{L}_{\text{CQL}}(\theta) = \alpha\, \mathbb{E}_{s \sim \mathcal{D}}\Big[ \log \sum_{a} e^{Q_\theta(s,a)} - \mathbb{E}_{a \sim \pi_\beta(\cdot \mid s)}\big[ Q_\theta(s,a) \big] \Big] + \frac{1}{2}\, \mathbb{E}_{(s,a,r,s') \sim \mathcal{D}}\Big[ \big( y - Q_\theta(s,a) \big)^2 \Big]$

The second term is the Double DQN loss we already have (the factor $\tfrac{1}{2}$ is the paper's convention; our code omits it, which only rescales $\alpha$). The first is the conservative penalty, with a coefficient $\alpha$ (again not a learning rate). To understand it, look at its gradient with respect to $Q_\theta(s,a)$. The log-sum-exp is a soft maximum over the actions, and its derivative with respect to $Q(s,a)$ is the softmax probability $e^{Q(s,a)} / \sum_b e^{Q(s,b)}$: the term pushes down the value of **every** action, in proportion to how high the network currently rates it, that is in proportion to how likely the greedy policy is to pick it. The second term pushes up the value of the action **in the data**. For an in-distribution action the two effects largely cancel; for an action that the network rates highly but that the data do not contain, nothing cancels the push-down. The net effect is a Q-function that **underestimates** the value of out-of-distribution actions, so that the greedy policy, and the maximum in the target, avoid them: the agent is confined to the region of the state-action space where the data can vouch for its estimates. Kumar and colleagues prove that, for $\alpha$ large enough, the value that the learned Q-function assigns to the learned policy, $\mathbb{E}_{a \sim \pi}[Q_\theta(s,a)]$, is a lower bound on the true value $v_\pi(s)$ of that policy (not action by action: individual Q-values of well-supported actions can be raised), which is why the method is called conservative.

The coefficient $\alpha$ controls the trade-off: with $\alpha = 0$ we are back to offline DQN and its extrapolation error; with $\alpha$ very large the agent stays so close to the data that it merely imitates them, and CQL degenerates into a form of behaviour cloning. In between, CQL can **improve on the behaviour policy** where the data contain good actions, which is the whole point. The implementation is a single line added to the loss, which is why we built the option into the function above: with the penalty active, the soft maximum is computed by `logsumexp` over the network's outputs for the sampled states:

In [ ]:
torch.manual_seed(34)
np.random.seed(34)
random.seed(34)

conservative_alpha = 1.0

cql_scores = {}
cql_logs = {}
for name, dataset in (("expert", expert_dataset), ("medium", medium_dataset), ("random", random_dataset)):
    cql_pi, q_log, max_q_log, data_return = offline_dqn(dataset, conservative_alpha=conservative_alpha)
    cql_scores[name] = evaluate(lunar_lander, cql_pi, episodes=20)
    cql_logs[name] = (q_log, max_q_log, data_return)
    print(f"\nCQL on the {name} dataset: score {cql_scores[name]:.1f}, Q of dataset actions {q_log[-1]:.1f}, Q of greedy actions {max_q_log[-1]:.1f}, dataset return {data_return:.1f}")

In [ ]:
plot_q_values(cql_logs, "CQL: estimated values vs. observed discounted returns")

The conservative penalty changes the picture completely. The value of the greedy actions no longer runs away: it stays within a hair of the value of the dataset actions, and both grow slowly towards values of the order of the observed returns instead of past them, consistent with a policy that is somewhat better than the behaviour policy but confined to actions the data support. The policies are now usable: on the medium dataset, CQL learns a policy that lands more often than the behaviour policy, by picking, in each state, the actions that the data show to lead to good outcomes and avoiding the ones they do not vouch for. On the random dataset there is little to learn from, and on the expert dataset the data are so narrow that cloning is already near-optimal, so the gains are smaller or absent. We can summarise the three methods on the three datasets:

In [ ]:
print(f"{'dataset':10s} {'behaviour':>10s} {'cloning':>10s} {'offline DQN':>12s} {'CQL':>10s}")
for name, behaviour in (("expert", expert_score), ("medium", medium_score), ("random", random_score)):
    print(f"{name:10s} {behaviour:10.1f} {bc_scores[name]:10.1f} {dqn_scores[name]:12.1f} {cql_scores[name]:10.1f}")

The table is the typical picture of offline reinforcement learning. Behaviour cloning is a strong baseline when the data are good and the task is to reproduce them; naive off-policy learning is unreliable whatever the data; a conservative method is the only one that can **exceed** the data when they contain the ingredients of a better policy, as the medium dataset does, with its mix of good and bad decisions. The ability to combine the good parts of mediocre trajectories into a better policy, sometimes called **stitching**, is what distinguishes offline reinforcement learning from imitation, and it is what the bootstrapped value function, used conservatively, provides.

## Decision Transformers

CQL keeps the machinery of value-based reinforcement learning and patches its weak point. The [**Decision Transformer**](./papers/2021%20-%20Decision%20Transformer%20-%20Reinforcement%20Learning%20via%20Sequence%20Modeling.pdf), introduced by Chen and colleagues in 2021, takes a radically different route. It observes that a dataset of trajectories is a collection of **sequences**, that the most powerful tool we have for learning from sequences is the **transformer** that underlies large language models, and that if the sequences are written down in the right way, the question "what action leads to a high return?" becomes the question "what token comes next?". There are no value functions, no Bellman equations, no bootstrapping, and therefore **no extrapolation error**: the model is trained by supervised learning, like behaviour cloning, but unlike behaviour cloning it is told, for every action, what return followed it, and at test time it can be asked for a high one.

### Self-Attention and the Transformer

Since transformers will also be the subject of Chapter 13, it is worth understanding the architecture rather than treating it as a black box. A transformer processes a sequence of $L$ **tokens**, each represented by an embedding vector $x_i \in \mathbb{R}^{E}$. The core operation is **self-attention**: every token builds a new representation of itself as a weighted average of the representations of the other tokens, with weights that depend on the content. Three linear maps produce, for each token, a **query** $q_i = W_Q x_i$, a **key** $k_i = W_K x_i$ and a **value** $v_i = W_V x_i$. The attention weight that token $i$ gives to token $j$ is the softmax, over $j$, of the scaled dot product between the query of $i$ and the key of $j$,

$\displaystyle w_{ij} = \frac{ \exp\big( q_i^\top k_j / \sqrt{d} \big) }{ \sum_{j'} \exp\big( q_i^\top k_{j'} / \sqrt{d} \big) }, \qquad z_i = \sum_{j} w_{ij}\, v_j$

so that each output $z_i$ gathers the values of the tokens whose keys match its query. In matrix form, with the queries, keys and values stacked in the rows of $\mathbf{Q}, \mathbf{K}, \mathbf{V}$ (bold, to distinguish them from action values), $\mathbf{Z} = \text{softmax}\big( \mathbf{Q} \mathbf{K}^\top / \sqrt{d} \big) \mathbf{V}$, where $d$ is the dimension of the keys. Several attention **heads** compute this in parallel with different weight matrices and concatenate their outputs, so that different heads can attend to different kinds of relations. For sequence *prediction* the attention must be **causal**: token $i$ may attend only to tokens $j \le i$, which is enforced by setting the weights of future tokens to zero (adding $-\infty$ to their scores before the softmax). A transformer **block** applies self-attention followed by a small feed-forward network to each token, each preceded by layer normalisation and wrapped in a residual connection; a transformer stacks several such blocks. Since attention treats the tokens as a set (permuting the inputs permutes the outputs in the same way), information about *where* each token sits in the sequence must be added to its embedding, as a **positional embedding**. This architecture, with causal attention and an output layer that predicts the next token, is the GPT architecture of language models.

### Trajectories as Sequences

The key design decision of the Decision Transformer is how to write a trajectory as a sequence of tokens. Writing states and actions alone would give behaviour cloning with memory. The Decision Transformer adds, before each state, the **return-to-go**, the sum of the rewards from that time step to the end of the episode:

$\displaystyle \hat R_t = \sum_{t' = t}^{T-1} R_{t'+1}, \qquad \tau = \big( \hat R_0, S_0, A_0,\ \hat R_1, S_1, A_1,\ \ldots,\ \hat R_{T-1}, S_{T-1}, A_{T-1} \big)$

Each of the three kinds of element is embedded by its own linear layer (an embedding table for discrete actions) into a vector of size $E$, a learned embedding of the **time step** $t$ is added to the three tokens of step $t$ as positional information, and the resulting sequence is processed by a causal transformer. The model is trained to **predict the action** $A_t$ from the output at the position of the state token $S_t$, which, thanks to the causal mask, has seen $\hat R_0, S_0, A_0, \ldots, \hat R_t, S_t$ and nothing later. For discrete actions the loss is the cross-entropy, exactly as in behaviour cloning. The model therefore learns the conditional distribution

$\displaystyle P\big( A_t \mid \hat R_t, S_t, \text{and the previous } K-1 \text{ steps} \big)$

of the action taken in a state, **given the return that followed it**. On the medium dataset, where some trajectories end in a landing and some in a crash, the model learns that high values of $\hat R$ go with one kind of action and low values with another. In practice, training uses sub-sequences of a fixed length of $K$ steps (the **context**), sampled at random from the trajectories, rather than whole episodes.

### Acting with a Desired Return

At test time the model is **prompted** with a desired return. We choose a target $\hat R_0$, for instance the return of a good landing, observe $S_0$, and ask the model for the action; after taking it and receiving $R_1$, the return-to-go is updated to $\hat R_1 = \hat R_0 - R_1$, the new state is appended, and the model is asked again, with the last $K$ steps as context. The model behaves as the trajectories in the data that achieved the requested return behaved, and, if the request is consistent with what the data can support, obtains something close to it. The desired return is a **knob**: asking for a low return should produce a bad policy, asking for a high one a good policy, and the achieved return should follow the target, up to the best that the dataset contains. This is a kind of capability that no other algorithm in this course has, and it comes essentially for free from the sequence formulation.

The model is implemented with the transformer blocks provided by PyTorch, with a causal mask. The forward pass takes the three sequences of a batch of $K$-step windows, interleaves their embeddings into a sequence of $3K$ tokens, and returns the action logits at the state positions:

In [ ]:
class DecisionTransformer(torch.nn.Module):

    def __init__(self, state_size, action_size, max_timesteps=1000, embedding_size=64, n_layers=3, n_heads=4, dropout=0.1):
        super().__init__()

        # embeddings of the three kinds of token, and of the time step
        self.embed_return = torch.nn.Linear(1, embedding_size)
        self.embed_state = torch.nn.Linear(state_size, embedding_size)
        self.embed_action = torch.nn.Embedding(action_size, embedding_size)
        self.embed_timestep = torch.nn.Embedding(max_timesteps + 1, embedding_size)

        # a stack of causal transformer blocks
        block = torch.nn.TransformerEncoderLayer(
            d_model=embedding_size, nhead=n_heads, dim_feedforward=4 * embedding_size,
            dropout=dropout, batch_first=True, norm_first=True,
        )
        self.transformer = torch.nn.TransformerEncoder(block, num_layers=n_layers)
        self.norm = torch.nn.LayerNorm(embedding_size)

        # output layer: the logits of the next action
        self.head = torch.nn.Linear(embedding_size, action_size)

    def forward(self, returns_to_go, states, actions, timesteps):

        batch_size, context = states.shape[0], states.shape[1]

        # embed the tokens and add the time-step embedding to each of them
        time_embedding = self.embed_timestep(timesteps)
        return_tokens = self.embed_return(returns_to_go.unsqueeze(-1)) + time_embedding
        state_tokens = self.embed_state(states) + time_embedding
        action_tokens = self.embed_action(actions) + time_embedding

        # interleave the tokens: (R_0, S_0, A_0, R_1, S_1, A_1, ...)
        tokens = torch.stack([return_tokens, state_tokens, action_tokens], dim=2)
        tokens = tokens.reshape(batch_size, 3 * context, -1)

        # causal mask: each token attends only to the tokens before it
        mask = torch.nn.Transformer.generate_square_subsequent_mask(3 * context)
        hidden = self.transformer(tokens, mask=mask, is_causal=True)
        hidden = self.norm(hidden)

        # the outputs at the state tokens predict the actions
        state_outputs = hidden.reshape(batch_size, context, 3, -1)[:, :, 1]

        return self.head(state_outputs)

Training samples windows of $K$ consecutive steps from the trajectories of the dataset. For each window we need the states, the actions, the time steps and the returns-to-go, which we compute once per episode. The returns are divided by a constant to bring them to the scale of the other embeddings:

In [ ]:
def returns_to_go(dataset):

    # undiscounted return-to-go of every transition, computed backwards within each episode
    rtg = np.zeros_like(dataset["rewards"])
    starts = dataset["episode_starts"]

    for episode in range(len(starts) - 1):
        future_return = 0.0
        for t in reversed(range(starts[episode], starts[episode + 1])):
            future_return = dataset["rewards"][t] + future_return
            rtg[t] = future_return

    return rtg


def sample_windows(dataset, rtg, context, batch_size, return_scale):

    starts = dataset["episode_starts"]
    lengths = np.diff(starts)

    # only episodes at least as long as the context are used; longer episodes are sampled more often
    usable = np.where(lengths >= context)[0]
    probabilities = lengths[usable] / lengths[usable].sum()

    batch_returns, batch_states, batch_actions, batch_timesteps = [], [], [], []

    for _ in range(batch_size):

        # a random episode and a random window inside it
        episode = np.random.choice(usable, p=probabilities)
        offset = np.random.randint(0, lengths[episode] - context + 1)
        begin = starts[episode] + offset

        batch_returns.append(rtg[begin:begin + context] / return_scale)
        batch_states.append(dataset["states"][begin:begin + context])
        batch_actions.append(dataset["actions"][begin:begin + context])
        batch_timesteps.append(np.arange(offset, offset + context))

    return (
        torch.as_tensor(np.array(batch_returns), dtype=torch.float32),
        torch.as_tensor(np.array(batch_states)),
        torch.as_tensor(np.array(batch_actions)),
        torch.as_tensor(np.array(batch_timesteps)),
    )

The training loop is plain supervised learning: sample a batch of windows, predict the actions, minimise the cross-entropy:

In [ ]:
def train_decision_transformer(dataset, iterations=5000, context=20, batch_size=64, learning_rate=0.0003, return_scale=100.0):

    model = DecisionTransformer(state_size, action_size)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=0.0001)

    rtg = returns_to_go(dataset)
    model.train()

    for iteration in range(1, iterations + 1):

        returns, states, actions, timesteps = sample_windows(dataset, rtg, context, batch_size, return_scale)

        # predict every action of the window from the tokens before it
        logits = model(returns, states, actions, timesteps)
        loss = torch.nn.functional.cross_entropy(logits.reshape(-1, action_size), actions.reshape(-1))

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.25)
        optimizer.step()

        if iteration % 500 == 0:
            print(f"Iteration {iteration:05}, loss {loss.item():.4f}", end="\r", flush=True)

    model.eval()
    return model

To act, the model needs the history of the current episode. The policy function below keeps that history, feeds the last $K$ steps to the model (with a placeholder for the action of the current step, which the causal mask hides from the state token anyway), takes the most probable action, and decrements the return-to-go by the reward received, which the environment loop must pass back to it. Since our evaluation function only calls `policy(state)`, we write a small evaluation loop for the Decision Transformer that handles the return-to-go:

In [ ]:
def evaluate_decision_transformer(env, model, target_return, episodes=20, context=20, return_scale=100.0):

    scores = []

    for episode in range(episodes):

        state, _ = env.reset()
        done = False
        total_reward = 0.0

        # the history of the episode
        returns, states, actions = [target_return], [state], []
        t = 0

        while not done:

            # the last K steps of the history, with a placeholder action for the current step
            window = slice(max(0, t - context + 1), t + 1)
            with torch.no_grad():
                logits = model(
                    torch.as_tensor(np.array(returns[window]) / return_scale, dtype=torch.float32).unsqueeze(0),
                    torch.as_tensor(np.array(states[window]), dtype=torch.float32).unsqueeze(0),
                    torch.as_tensor(np.array(actions[window] + [0])).unsqueeze(0),
                    torch.as_tensor(np.arange(t + 1)[window]).unsqueeze(0),
                )
            action = int(torch.argmax(logits[0, -1]).item())

            # act, and update the history: the return-to-go decreases by the reward received
            state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            total_reward += reward

            actions.append(action)
            returns.append(returns[-1] - reward)
            states.append(state)
            t += 1

        scores.append(total_reward)

    return float(np.mean(scores))

We train a Decision Transformer on the medium dataset, whose trajectories range from crashes to good landings, and then ask it for different returns:

In [ ]:
torch.manual_seed(34)
np.random.seed(34)
random.seed(34)

decision_transformer = train_decision_transformer(medium_dataset, iterations=5000)

In [ ]:
target_returns = [-100, 0, 100, 200, 300]
achieved_returns = []

for target in target_returns:
    achieved = evaluate_decision_transformer(lunar_lander, decision_transformer, target_return=target, episodes=20)
    achieved_returns.append(achieved)
    print(f"Target return {target:5d}: achieved return {achieved:7.1f}")

In [ ]:
plt.figure(figsize=(8, 6))
plt.title("Decision Transformer on the medium dataset: achieved vs. target return")
plt.xlabel("Target return")
plt.ylabel("Achieved return")
plt.plot(target_returns, achieved_returns, "o-", color="purple", linewidth=2, label="Decision Transformer")
plt.plot(target_returns, target_returns, "--", color="gray", linewidth=1, label="achieved = target")
plt.axhline(medium_dataset["episode_returns"].max(), color="orange", linestyle=":", linewidth=1, label="best return in the dataset")
plt.legend()
plt.show()

The achieved return **follows the target**: asked for a crash, the model crashes; asked for a modest return, it hovers and comes down somewhere; asked for a good landing, it lands, with a success rate that increases with the target. The same network, trained once by supervised learning, contains a whole family of policies indexed by the desired return. The correspondence holds in the range of returns that the data represent well; near and beyond the best returns in the dataset, which only a handful of trajectories achieved, the curve flattens or even drops: the model can only reproduce behaviours it has seen, and asking for more than the data contain is asking for extrapolation, which a sequence model resolves by imitating whatever rare trajectories resemble the request, with unreliable results.

This last point is both the strength and the weakness of the approach. A Decision Transformer never bootstraps, so it never suffers from extrapolation error, and it is trained with the stable, well-understood machinery of supervised learning on sequences, which scales to very large models and datasets. On the other hand, it has no mechanism to **stitch** together the good parts of different trajectories into a trajectory better than any in the data, which is exactly what a value function offers: it learns which actions *went with* high returns, not which actions *cause* them. In practice the two families, conservative value learning and return-conditioned sequence modelling, are complementary, and which one works better depends on the dataset.

## Limitations and Outlook

Offline reinforcement learning removes the last assumption of the course, the possibility of interacting with the environment, and in doing so it exposes the foundations of everything that came before. Value-based methods bootstrap, and bootstrapping extrapolates; online, exploration pays for the extrapolation by testing it, offline nothing does, and the estimates must be kept conservative by design. Supervised approaches avoid the problem but can only recombine what the data contain, and conditioning on the return turns imitation into something closer to control. Three difficulties remain open and define the current research.

First, **evaluation**: we measured our policies in the simulator, which a real offline problem forbids. Estimating the value of a new policy from logged data, off-policy evaluation, is itself hard, and model selection, choosing between two offline agents without running them, is often the practical bottleneck.

Second, **coverage**: every method in this chapter is limited by what the dataset contains. A conservative agent cannot improve where the data show no better action, a Decision Transformer cannot exceed the best trajectory it has seen, and neither can tell, from the data alone, how far it is from the optimum. The quality of an offline agent is first of all the quality and diversity of its dataset.

Third, the **choice of approach**. CQL and its successors keep the value function and tame it; Decision Transformers and their successors keep the sequence model and make it more capable, for instance by training on data from many tasks at once. The second family connects directly to the next chapter: a language model is a causal transformer trained on sequences, and the fine-tuning of language models from human preferences has a strong offline flavour: the preference data are collected once, the reward model is fitted to them like a supervised model, and the policy must not drift far from the data that produced it, which is why some of its variants dispense with interaction altogether.